# Study 09 · 10 · Summary of results

Collects the outputs of notebooks 01–09 (`models/experimentos/revision_epidemics/resultados/`) into one table per question and writes the headline sentences directly from the numbers, so text and results cannot drift apart. Conventions: 18 seasons (2009–2026), pooled over district-weeks, median over seeds; 95% whole-season bootstrap intervals; negative Brier change favours the first arm.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import ast
RES = c.OUTPUT / "resultados"
read = lambda name: pd.read_csv(RES / name)
def ci(text):
    lo, hi = ast.literal_eval(text) if isinstance(text, str) else text
    return f"[{float(lo):+.2f}, {float(hi):+.2f}]"
def fmt(row, field="cambio_agrupado_%", interval="ic95_agrupado"):
    return f"{row[field]:+.2f}% {ci(row[interval])}"


## 1 · Headline numbers

In [2]:
e1 = read("e1_principal_18_temporadas.csv")
climate = e1[e1.comparacion.str.startswith("b3 vs history")].set_index("horizonte")
pp = e1[e1.comparacion.eq("b3 vs probabilistic persistence")].set_index("horizonte")
hgpp = e1[e1.comparacion.eq("history+geography vs probabilistic persistence")].set_index("horizonte")
lags = read("e2_rezagos.csv"); lags18 = lags[lags.seasons.eq("18 seasons")]
lag = {a: lags18[lags18.arm.eq(a)].set_index("horizonte") for a in lags18.arm.unique()}
plac = read("e2_placebos.csv")
dec = read("e5_descomposicion.csv")
hist = dec[dec.contrast.eq("adding case history, no climate")].set_index("horizonte")
clim_h = dec[dec.contrast.eq("adding climate, with case history")].set_index("horizonte")
cal = read("e4_contrastes.csv")
nb2 = read("e7_nb2.csv")
lines = [
    f"E1 · Adding the b3 climate block to case history + geography changed Brier by {fmt(climate.loc[2])}, {fmt(climate.loc[3])} and {fmt(climate.loc[4])} at 2, 3 and 4 weeks (18 seasons, pooled). Equal-season means: {climate.loc[2, 'cambio_igual_peso_%']:+.2f}%, {climate.loc[3, 'cambio_igual_peso_%']:+.2f}%, {climate.loc[4, 'cambio_igual_peso_%']:+.2f}%. Climate was better in {int(climate.loc[4, 'temporadas_a_mejor'])} of 18 seasons at 4 weeks.",
    f"E1 · Against calibrated persistence (two parameters per horizon), b3 changed Brier by {fmt(pp.loc[4])} at 4 weeks pooled, but by {pp.loc[4, 'cambio_igual_peso_%']:+.2f}% with equal season weights; it beat persistence in {int(pp.loc[4, 'temporadas_a_mejor'])} of 18 seasons. History + geography: {fmt(hgpp.loc[4])} pooled.",
    f"E2 · Four-week Brier change vs history + geography by weather lag: lag 0 (look-ahead) {fmt(lag['lag 0 (historical, look-ahead)'].loc[4])}; lag 1 {fmt(lag['lag 1 (issue time)'].loc[4])}; lag 2 {fmt(lag['lag 2 (issue time, paper)'].loc[4])}.",
    f"E2 · Observed vs wrong-year temperature (median of three placebos, 4 weeks): {plac[(plac.placebo.str.startswith('median')) & plac.horizonte.eq(4)]['cambio_agrupado_%'].iloc[0]:+.2f}%; vs all-variables placebo: {fmt(plac[(plac.placebo.eq('b3_placebo_conjunto')) & plac.horizonte.eq(4)].iloc[0])}.",
    f"E5 · Adding case history (no climate) changed Brier by {hist.loc[2, 'cambio_agrupado_%']:+.1f}%, {hist.loc[3, 'cambio_agrupado_%']:+.1f}% and {hist.loc[4, 'cambio_agrupado_%']:+.1f}% (better in all 18 seasons); adding climate to case history (no offset formulation): {fmt(clim_h.loc[4])} at 4 weeks.",
    "E4 · Climate contrast at 4 weeks by calibration: " + "; ".join(f"{v}: {fmt(r)}" for v, r in cal[cal.horizonte.eq(4)].set_index('variant').iterrows()) + ".",
    f"E7 · NB2 regression, adding the b3 block: {fmt(nb2[nb2.comparison.str.startswith('NB2 + b3') & nb2.horizonte.eq(4)].iloc[0])} at 4 weeks.",
    "E8 · Climate effect at longer horizons: " + "; ".join(f"{int(h)} weeks {fmt(r)}" for h, r in
        read("e8_principal.csv").query("comparacion.str.startswith('b3 vs history')", engine="python").set_index("horizonte").iterrows()) + ".",
]
for line in lines:
    print("•", line, "\n")

• E1 · Adding the b3 climate block to case history + geography changed Brier by -0.43% [-1.46, +0.37], -0.28% [-1.89, +0.87] and +0.04% [-2.31, +1.70] at 2, 3 and 4 weeks (18 seasons, pooled). Equal-season means: -1.31%, -1.39%, -1.15%. Climate was better in 10 of 18 seasons at 4 weeks. 

• E1 · Against calibrated persistence (two parameters per horizon), b3 changed Brier by -4.76% [-9.18, +2.38] at 4 weeks pooled, but by +8.62% with equal season weights; it beat persistence in 9 of 18 seasons. History + geography: -4.80% [-10.08, +4.02] pooled. 

• E2 · Four-week Brier change vs history + geography by weather lag: lag 0 (look-ahead) -1.73% [-3.67, -0.22]; lag 1 -0.38% [-2.52, +1.16]; lag 2 +0.04% [-2.31, +1.70]. 

• E2 · Observed vs wrong-year temperature (median of three placebos, 4 weeks): +0.13%; vs all-variables placebo: -0.46% [-1.95, +0.56]. 

• E5 · Adding case history (no climate) changed Brier by -57.7%, -50.6% and -44.6% (better in all 18 seasons); adding climate to case his

## 2 · Data and outcome definition (E6)

In [3]:
for name in ("e6b_confirmados_probables.csv", "e6c_umbral_sin_epidemias.csv", "e6d_incidencia.csv"):
    t = read(name)
    t = t[t.horizonte.eq(4)]
    print(name)
    display(t.drop(columns=[x for x in ("ic95_igual_peso",) if x in t]).round(3))

e6b_confirmados_probables.csv


,seasons,comparison,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
2,2009-2025 (protocol),b3 vs history+geography,4,0.037,0.038,-0.832,"(-3.07, 1.04)",-0.884,11
5,2009-2025 (protocol),b3 vs probabilistic persistence,4,0.037,0.040,-6.360,"(-11.19, 1.46)",9.095,10
8,2009-2026,b3 vs history+geography,4,0.036,0.036,-0.845,"(-3.0, 1.0)",-0.927,12
11,2009-2026,b3 vs probabilistic persistence,4,0.036,0.038,-5.991,"(-10.89, 1.59)",9.385,10


e6c_umbral_sin_epidemias.csv


,horizonte,temporadas,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor,comparison
2,4,18,0.045,0.045,-0.376,"(-3.26, 1.81)",-0.112,11,b3 vs history+geography
5,4,18,0.045,0.045,-1.398,"(-7.27, 7.59)",13.777,10,b3 vs probabilistic persistence


e6d_incidencia.csv


,horizonte,temporadas,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor,comparison
2,4,7,0.058,0.057,2.298,"(-1.1, 5.93)",7.099,1,b3 vs history+geography
5,4,7,0.058,0.057,2.910,"(-13.75, 40.52)",15.146,3,b3 vs probabilistic persistence


## 3 · Diagnostics on the fixed-hyperparameter models (E3)

In [4]:
probability = read("e3_diagnostico_probabilidades.csv")
fields = ["brier", "mcb", "dsc", "ap", "pendiente", "intercepto"]
display(probability.groupby(["brazo", "horizonte", "temporada"])[fields].median().groupby(["brazo", "horizonte"]).mean().round(5))
counts = read("e3_conteos.csv")
summary = counts.groupby(["brazo", "horizonte", "temporada"]).median(numeric_only=True).groupby(["brazo", "horizonte"]).mean()
summary["mae_ratio_vs_persistence"] = summary.mae / summary.mae_persistencia
summary[["mae", "rmse", "wis", "mae_ratio_vs_persistence"]].round(3)

brier      mcb      dsc       ap  pendiente  intercepto
brazo                     horizonte                                                           
b3                        2          0.02824  0.00300  0.03722  0.62048    1.09440    -0.37900
                          3          0.03220  0.00359  0.03385  0.55817    1.14089    -0.28949
                          4          0.03562  0.00435  0.03116  0.50313    1.14370    -0.38940
climatology               2          0.06973  0.01552  0.00823  0.14388    0.16384    -2.46365
                          3          0.06975  0.01550  0.00819  0.14370    0.16705    -2.42432
                          4          0.06975  0.01550  0.00819  0.14400    0.16770    -2.41399
history+geography         2          0.02835  0.00310  0.03719  0.62035    1.07552    -0.48144
                          3          0.03227  0.00365  0.03386  0.55863    1.08999    -0.51064
                          4          0.03560  0.00436  0.03120  0.50148    1.12996    -0.45489
probabilistic persistence 2          0.02863  0.00192  0.03573  0.58437    1.10123     0.06827
                          3          0.03325  0.00273  0.03192  0.52708    1.13593     0.11331
                          4          0.03742  0.00362  0.02864  0.48176    1.15443     0.14176

mae    rmse    wis  mae_ratio_vs_persistence
brazo             horizonte                                                
b3                2          1.730   8.961    NaN                     0.901
                  3          2.177  11.502    NaN                     0.867
                  4          2.560  13.454  1.888                     0.851
history+geography 2          1.801   9.429    NaN                     0.938
                  3          2.323  12.279    NaN                     0.925
                  4          2.781  14.653  1.967                     0.924

## 4 · Figures\n\nSaved in `research/09_epidemics_revision/figures/`: `e1_contraste_por_temporada.png`, `e2_rezagos.png`, `e3_beneficio_neto.png`, `e5_que_aporta_la_habilidad.png`, `e8_por_horizonte.png`.